In [2]:
import time
from typing import Any, List, Tuple
from playwright.sync_api import sync_playwright

from google import genai
from google.genai import types
from google.genai.types import Content, Part

from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
with open("initial_screenshot.png", "rb") as f:
    initial_screenshot = f.read()

In [8]:
client = genai.Client()

In [6]:
USER_PROMPT = "Go to ai.google.dev/gemini-api/docs and search for pricing."
print(f"Goal: {USER_PROMPT}")  

config = types.GenerateContentConfig(
    tools=[types.Tool(computer_use=types.ComputerUse(
        environment=types.Environment.ENVIRONMENT_BROWSER
    ))],
    thinking_config=types.ThinkingConfig(include_thoughts=True),
)

contents = [
    Content(role="user", parts=[
        Part(text=USER_PROMPT),
        Part.from_bytes(data=initial_screenshot, mime_type='image/png')
    ])
]  

Goal: Go to ai.google.dev/gemini-api/docs and search for pricing.


In [9]:
# First inference
response = client.models.generate_content(
    model='gemini-2.5-computer-use-preview-10-2025',
    contents=contents,
    config=config,
)

In [22]:
# response.model_dump()

In [10]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            )
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-computer-use-preview-10-2025',
  response_id='vRkTaoHNBq7K4-EPwr27wAI',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=12,
    prompt_token_count=1826,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=20
      ),
      ModalityTokenCount(
        modality=<MediaModality.IMAGE: 'IMAGE'>,
        token_count=1806
      ),
    ],
    total_token_count=1838
  )
)

In [13]:
response.candidates[0]

Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={},
          name='open_web_browser'
        )
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)

In [15]:
candidate = response.candidates[0]

In [16]:
has_function_calls = any(part.function_call for part in candidate.content.parts)

In [17]:
has_function_calls

True

In [19]:
results = []
function_calls = []
for part in candidate.content.parts:
    if part.function_call:
        function_calls.append(part.function_call)

for function_call in function_calls:
    action_result = {}
    fname = function_call.name
    args = function_call.args
    print(f"  -> Executing: {fname}")

  -> Executing: open_web_browser


### Turn 2